# v1 HaHaScore — Text-only RoBERTa Regression on Jester (Colab)

**Canonical path** per council realignment (Oct 5 2026).

- **Task**: continuous humor strength 0-100 from joke text
- **Model**: RoBERTa-base (frozen) + Linear(768, 1) regression head
- **Data**: SeppeV/rated_jokes_dataset_from_jester (1.76M rated jokes, Apache-2.0)
- **Validation**: 5×3 repeated joke-disjoint CV with Spearman ρ, MAE, RMSE + bootstrap 95% CI

**Pre-registered success gate**: ρ ≥ 0.35 AND bootstrap CI excludes 0.30.

**Source**: github.com/Das-rebel/HaHaScore (commit `f66933e`)

## Cell 1: Setup

In [ ]:
!pip install --quiet transformers==4.44.0 datasets==4.8.4 scipy scikit-learn numpy
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2: Clone repo + pull scripts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/Das-rebel/HaHaScore.git
%cd /content/HaHaScore
!git log --oneline -3
import sys; sys.path.insert(0, '/content/HaHaScore')
print('Scripts loaded:', [m for m in sys.modules if 'laptop' in m][:5])

## Cell 3: Download Jester (~5 min, 0 GPU)

In [ ]:
!python3 data/jester_seppev.py --output-dir data/jester
!ls -lh data/jester/

## Cell 4: Verify download + show sample

In [ ]:
import pandas as pd
df = pd.read_csv('data/jester/jester_full.csv')
print(f'Total rows: {len(df):,}')
print(f'Unique jokes: {df["joke_id"].nunique():,}')
print(f'Rating range: [{df["rating"].min():.2f}, {df["rating"].max():.2f}]')
print(f'Mean rating: {df["rating"].mean():.2f}')
print(f'\nSample rows:')
print(df.head(3).to_string())

## Cell 5: Smoke test (10k samples, 3 epochs, 3 seeds × 3 folds = 9 measurements)

In [ ]:
!python3 train_jester_regression.py \
    --max-samples 10000 \
    --epochs 3 \
    --n-seeds 3 \
    --n-folds 3 \
    --out-dir experiments/v1_jester_regression_smoke

## Cell 6: Full run (1.76M jokes, 5 epochs, 3 seeds × 5 folds = 15 measurements) — ~4-6h on T4

In [ ]:
!python3 train_jester_regression.py \
    --epochs 5 \
    --n-seeds 3 \
    --n-folds 5 \
    --out-dir experiments/v1_jester_regression

## Cell 7: Verify pre-registered gate

In [ ]:
import json
import glob
results = sorted(glob.glob('experiments/v1_jester_regression/result_*.json'))
latest = results[-1]
print(f'Loading: {latest}')
data = json.load(open(latest))
print(f'\n=== v1 Results ===')
print(f'Spearman rho: {data["rho_mean"]:.4f} ± {data["rho_std"]:.4f}')
print(f'95% CI:        [{data["rho_ci_low"]:.4f}, {data["rho_ci_high"]:.4f}]')
print(f'MAE:           {data["mae_mean"]:.4f}')
print(f'RMSE:          {data["rmse_mean"]:.4f}')
print(f'\n=== Pre-registered gates ===')
gates = data.get('pre_registered_gates', {})
rho_target = gates.get('spearman_rho_min', 0.35)
ci_target = gates.get('ci_lower_bound_min', 0.30)
rho_pass = data['rho_mean'] >= rho_target
ci_pass = data['rho_ci_low'] >= ci_target
print(f'  rho >= {rho_target}: {data["rho_mean"]:.4f} -> {"PASS" if rho_pass else "FAIL"}')
print(f'  ci >= {ci_target}: {data["rho_ci_low"]:.4f} -> {"PASS" if ci_pass else "FAIL"}')
print(f'\nVERDICT: {"✅ PUBLISHABLE" if rho_pass and ci_pass else "⚠️ NOT YET PUBLISHABLE"}')
if not (rho_pass and ci_pass):
    print('Next steps:')
    print('  1. Try larger model (roberta-large)')
    print('  2. Add more epochs')
    print('  3. Add LoRA r=8 fine-tuning (instead of frozen backbone)')

## Cell 8: Save results to Drive

In [ ]:
import shutil
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/HaHaScore/v1_jester_runs')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for f in glob.glob('experiments/v1_jester_regression/*.json'):
    shutil.copy(f, DRIVE_DIR / Path(f).name)
    print(f'Saved: {DRIVE_DIR / Path(f).name}')
print(f'\nAll results in: {DRIVE_DIR}')
print('Send back the result_*.json to commit to HaHaScore repo')